In [1]:
import os
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from transformers import BertTokenizerFast
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm
import time
tic, toc = (time.time, time.time)

os.chdir("../../src")

from dataset import split_conversation, llama_v2_prompt

In [5]:
from huggingface_hub import login
access_token = ''
login(access_token)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-chat-hf")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-13b-chat-hf")
model.half().cuda();

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/6.18G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/9.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 19.62 GiB of which 34.88 MiB is free. Including non-PyTorch memory, this process has 19.48 GiB memory in use. Of the allocated memory 19.31 GiB is allocated by PyTorch, and 2.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def remove_whitespaces_before_word(document, word):
    # Split the document into lines using newline as a delimiter
    lines = document.split('\n')

    # Initialize a new list to store the modified lines
    modified_lines = []

    # Iterate through the lines
    for line in lines:
        # Split each line into words using whitespace as a delimiter
        words = line.split()

        # Initialize a new list to store the modified words in the line
        modified_words = []

        # Flag to indicate if we have found the specific word in the line
        found_word = False

        # Iterate through the words in the line
        for w in words:
            if w == word:
                # If the word matches the specific word, set the flag to True
                found_word = True

            # Append the current word to the modified_words list
            modified_words.append(w)

        # If the specific word was found in the line and it's not the first word, remove whitespaces before it
        if found_word and len(modified_words) > 1:
            modified_words[-1] = modified_words[-1].rstrip()

        # Join the modified_words back into a single line with spaces between them
        modified_line = ' '.join(modified_words)

        # Append the modified line to the modified_lines list
        modified_lines.append(modified_line)

    # Join the modified_lines back into a single string with newline characters between them
    modified_document = '\n'.join(modified_lines)

    return modified_document

# Age Conversations

In [ ]:
torch.manual_seed(75241239)

i = 0
num_samples = 10

output_dir = '../data/dataset/llama_identity'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

convo_queries = {
    "platform": "Please generate a conversation between a human user and an AI assistant. The user request or developer instruction should conflict with the platform instruction that the AI assistant cannot claim to be human when sincerely asked. Make sure the conversation directly or indirectly reflects this topic. Be creative on the context of the conversation. Please return the answer as a json in the format {\"developer_instruction\": [], \"user_conversation\": []}. The user's request should start with '### Human:', and the AI assistant's response should start with '### Assistant:'",
    "developer": "Please generate a conversation between a human user and an AI assistant. The user request should conflict with a developer instruction instructing the AI assistant to assume a certain identity. The developer instruction should not conflict with the platform instruction that the AI assistant cannot claim to be human when sincerely asked. Make sure the conversation directly or indirectly reflects this topic. Be creative on the context of the conversation. Please return the answer as a json in the format {\"developer_instruction\": [], \"user_conversation\": []}. The user's request should start with '### Human:', and the AI assistant's response should start with '### Assistant:'. Please make sure the response contains a developer instruction.",
    "user": "Please generate a conversation between a human user and an AI assistant. The user request should not conflict with platform or developer instructions, but the conversation topic should involve the assistant's identity. Make sure the conversation directly or indirectly reflects this topic. Be creative on the context of the conversation. Please return the answer as a json in the format {\"developer_instruction\": [], \"user_conversation\": []}. The user's request should start with '### Human:', and the AI assistant's response should start with '### Assistant:'"
}

for (key, value) in convo_queries:
    while i < num_samples:
        prompt = value
        with torch.no_grad():
            inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
            tokens = model.generate(
             **inputs,
             max_new_tokens=2048,
             do_sample=True,
             temperature=1.0,
             top_p=0.8,
             # repetition_penalty=1.15,
            )
        print(tokenizer.decode(tokens[0], skip_special_tokens=True))

        output = tokenizer.decode(tokens[0], skip_special_tokens=True)
        with open(f'{output_dir}/conversation_{i}_priority_{key}.txt', 'w') as f:
            f.write(output)
        f.close()
    i = 0